# 🔢 Notebook 2 — Embeddings + Indexação Qdrant

Implementa as Etapas 7 e 8 do pipeline:

- **Embeddings:** `intfloat/multilingual-e5-large-instruct` (1024 dims) — recomendação do PDF
- **Indexação:** Qdrant local com payload completo (texto_pai, ato_id, tipo_documento, situação)
- **Filtro nativo por tipo_documento** resolve o problema de distribuição (53% votos+notas)
- **Persistência em disco** — não recarrega vetores na memória entre execuções

**Pré-requisito:** Notebook 1 executado, `chunks_hierarquicos.parquet` no Drive

**⚠️ Ative GPU:** `Runtime > Change runtime type > T4 GPU` (~25min com GPU; ~8h em CPU)

In [ ]:
# ============================================================
# CÉLULA 1 — Instalação
# ============================================================
%%capture
!pip install sentence-transformers qdrant-client tqdm pyarrow

In [ ]:
# ============================================================
# CÉLULA 2 — Imports e configs
# ============================================================
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, PayloadSchemaType,
)

# === CONFIGURAÇÕES ===
EMBEDDING_MODEL  = 'intfloat/multilingual-e5-large-instruct'
VECTOR_DIM       = 1024  # e5-large-instruct
BATCH_SIZE       = 64    # Reduza para 32 se faltar VRAM
COLLECTION_NAME  = 'aneel_legislacao'

DRIVE_DIR    = '/content/drive/MyDrive/aneel_rag'
QDRANT_PATH  = '/content/qdrant_storage'
EMB_CHECKPOINT = '/content/embeddings_checkpoint.npy'
EMB_INDEX_FILE = '/content/embeddings_idx.txt'

# Verifica GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device.upper()}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('   ⚠️  Sem GPU. Ative em Runtime > Change runtime type > T4 GPU')

In [ ]:
# ============================================================
# CÉLULA 3 — Carrega chunks do Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print('📥 Carregando chunks do Drive...')
df_chunks = pd.read_parquet(f'{DRIVE_DIR}/chunks_hierarquicos.parquet')
print(f'✅ {len(df_chunks):,} chunks carregados')
print(f'   Colunas: {list(df_chunks.columns)}')

In [ ]:
# ============================================================
# CÉLULA 4 — Carrega modelo de embeddings
# ============================================================
print(f'📥 Carregando {EMBEDDING_MODEL}...')
print('   (~2GB de download na primeira vez)')
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
embed_model.max_seq_length = 384  # cobre os chunks filho com folga
print(f'✅ Modelo carregado | dim={embed_model.get_sentence_embedding_dimension()}')

# Validação de dimensionalidade
actual_dim = embed_model.get_sentence_embedding_dimension()
assert actual_dim == VECTOR_DIM, f'Dim esperado {VECTOR_DIM}, real {actual_dim}'

In [ ]:
# ============================================================
# CÉLULA 5 — Gera embeddings (com checkpoint a cada 10k)
# ============================================================
# e5-large-instruct usa formato:
#   passage: <texto>      ← para documentos
#   query: Instruct: <task>\nQuery: <pergunta>  ← para queries (no notebook 3)

# Texto a indexar = contexto_juridico + texto_filho (enriquece embedding)
def make_passage(row):
    ctx = row['contexto_juridico']
    txt = row['texto']
    return f'passage: [{ctx}] {txt}' if ctx else f'passage: {txt}'

passages = df_chunks.apply(make_passage, axis=1).tolist()
total = len(passages)

# Verifica checkpoint
start_idx = 0
all_embs = []
if os.path.exists(EMB_CHECKPOINT) and os.path.exists(EMB_INDEX_FILE):
    saved = np.load(EMB_CHECKPOINT)
    with open(EMB_INDEX_FILE) as f:
        start_idx = int(f.read().strip())
    all_embs.append(saved)
    print(f'♻️  Retomando do checkpoint: {start_idx:,}/{total:,}')

print(f'🚀 Gerando embeddings para {total - start_idx:,} chunks (batch={BATCH_SIZE})')
t0 = time.time()

for i in tqdm(range(start_idx, total, BATCH_SIZE), desc='Embeddings', unit='batch'):
    batch = passages[i : i + BATCH_SIZE]
    with torch.no_grad():
        emb = embed_model.encode(
            batch,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
    all_embs.append(emb.astype('float32'))

    # Checkpoint a cada ~10k chunks (resiliência contra desconexão Colab)
    if (i // BATCH_SIZE) % max(1, 10_000 // BATCH_SIZE) == 0 and i > start_idx:
        np.save(EMB_CHECKPOINT, np.vstack(all_embs))
        with open(EMB_INDEX_FILE, 'w') as f:
            f.write(str(i + BATCH_SIZE))

embeddings = np.vstack(all_embs).astype('float32')
elapsed = (time.time() - t0) / 60
print(f'\n✅ Embeddings gerados em {elapsed:.1f} min')
print(f'   Shape: {embeddings.shape} | RAM: {embeddings.nbytes / 1e6:.1f} MB')

# Salvar embeddings finais
np.save('/content/embeddings_final.npy', embeddings)

# Liberar GPU
del embed_model
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# CÉLULA 6 — Cria coleção Qdrant local
# ============================================================
# Qdrant em modo local — persiste em disco, sem servidor externo.
# Permite filtros por tipo_documento, situação, ano, ato_id durante o retrieval.

# Limpa storage antigo se existir (evita conflitos de schema)
if os.path.exists(QDRANT_PATH):
    os.system(f'rm -rf {QDRANT_PATH}')

client = QdrantClient(path=QDRANT_PATH)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)

# Cria índices nos campos de filtro (acelera queries com where)
for field, schema in [
    ('tipo_documento', PayloadSchemaType.KEYWORD),
    ('situacao',       PayloadSchemaType.KEYWORD),
    ('ato_id',         PayloadSchemaType.KEYWORD),
    ('ano',            PayloadSchemaType.INTEGER),
]:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=schema,
    )

print(f'✅ Coleção "{COLLECTION_NAME}" criada')
print(f'   Dimensões: {VECTOR_DIM} | Distância: COSINE')
print(f'   Índices: tipo_documento, situacao, ato_id, ano')

In [ ]:
# ============================================================
# CÉLULA 7 — Upload dos vetores para o Qdrant
# ============================================================
# Insere em batches de 256 para eficiência

UPSERT_BATCH = 256

print(f'📤 Inserindo {len(embeddings):,} pontos no Qdrant...')
t0 = time.time()

for i in tqdm(range(0, len(embeddings), UPSERT_BATCH), desc='Upsert'):
    batch_end = min(i + UPSERT_BATCH, len(embeddings))
    points = []
    for j in range(i, batch_end):
        row = df_chunks.iloc[j]
        # Payload mínimo necessário para retrieval e citação
        payload = {
            'texto':            row['texto'],
            'texto_pai':        row['texto_pai'],
            'ato_id':           row['ato_id'],
            'tipo_documento':   row['tipo_documento'],
            'titulo':           row['titulo'],
            'ementa':           row['ementa'],
            'assunto':          row['assunto'],
            'situacao':         row['situacao'],
            'publicacao':       row['publicacao'],
            'autor':            row['autor'],
            'ano':              int(row['ano']),
            'contexto_juridico': row['contexto_juridico'],
            'arquivo_origem':   row['arquivo_origem'],
        }
        points.append(PointStruct(
            id=int(j),
            vector=embeddings[j].tolist(),
            payload=payload,
        ))

    client.upsert(collection_name=COLLECTION_NAME, points=points)

elapsed = (time.time() - t0) / 60
info = client.get_collection(COLLECTION_NAME)
print(f'\n✅ Upsert concluído em {elapsed:.1f} min')
print(f'   Pontos indexados: {info.points_count:,}')
print(f'   Status: {info.status}')

In [ ]:
# ============================================================
# CÉLULA 8 — Teste de retrieval rápido
# ============================================================
# Validação: carrega o modelo de novo (só pra teste) e roda 1 query

print('🧪 Teste de retrieval...')
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)

# Formato query do e5-instruct
task = 'Given a question about Brazilian electric sector regulations, retrieve relevant legal passages'
query_text = 'Quais são os critérios para revisão tarifária das distribuidoras de energia?'
query_formatted = f'Instruct: {task}\nQuery: {query_text}'

q_emb = embed_model.encode(query_formatted, normalize_embeddings=True).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=q_emb,
    limit=5,
).points

print(f'\n🔍 Query: "{query_text}"')
for i, r in enumerate(results, 1):
    p = r.payload
    print(f"\n{i}. score={r.score:.4f} | {p['tipo_documento']} | {p['publicacao']}")
    print(f"   {p['ato_id']} | {p['titulo'][:80]}")
    print(f"   {p['texto'][:200]}...")

# Liberar memória
del embed_model
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# CÉLULA 9 — Salvar Qdrant no Drive (compactado)
# ============================================================
# Como o Qdrant ocupa muito (~3-5 GB), comprimimos antes de subir

# Importante: fechar o cliente antes de copiar (lock de arquivos)
client.close()
del client
gc.collect()

print('🗜️  Comprimindo storage do Qdrant...')
tar_path = '/content/qdrant_storage.tar.gz'
!tar -czf {tar_path} -C /content qdrant_storage
size_mb = os.path.getsize(tar_path) / 1e6
print(f'✅ Compactado: {size_mb:.1f} MB')

print('📤 Copiando para o Drive...')
!cp {tar_path} {DRIVE_DIR}/qdrant_storage.tar.gz

print(f'\n☁️  Arquivos no Drive:')
!ls -lh {DRIVE_DIR}

print('\n🎉 NOTEBOOK 2 CONCLUÍDO!')
print('   Próximo: 03_rag_pipeline.ipynb')